In [1]:
from ultralytics import YOLO

# Cargar el modelo preentrenado
model = YOLO('yolov8n.pt')

# # Entrenar
# model.train(
#     #data='coco8.yaml',
#     # data='coco128.yaml', 
#     epochs=3,
#     imgsz=640
# )


In [2]:
# Evaluar sobre el dataset correspondiente
#metrics = model.val(data="coco8.yaml", plots=True)
metrics = model.val(data="coco128.yaml", plots=True)

# Mostrar resultados 
print(metrics)

Ultralytics 8.3.204  Python-3.13.7 torch-2.8.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access  (ping: 0.30.2 ms, read: 40.930.5 MB/s, size: 49.3 KB)
val: Scanning C:\Users\menci\.spyder-py3\datasets\coco128\labels\train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 212.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 0.8it/s 9.7s1.4ss
                   all        128        929      0.639      0.536      0.607      0.448
                person         61        254      0.793      0.677      0.764      0.538
               bicycle          3          6      0.514      0.333      0.315      0.264
                   car         12         46      0.813      0.217      0.272      0.167
            motorcycle          4          5      0.687      0.887      0.898      0.718
    

In [5]:
cm = metrics.confusion_matrix.matrix

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

In [13]:
labels = list(model.names.values()) if hasattr(model, 'names') else [str(i) for i in range(cm.shape[0])]
labels.append("Extra")  # o "Unknown" según prefieras

In [14]:
print(len(labels))
print(cm.shape)


81
(81, 81)


In [15]:
# Dibujar el heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt=".0f", xticklabels=labels, yticklabels=labels, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.title("Matriz de Confusión YOLO11")
plt.tight_layout()
plt.show()

<Figure size 1200x1000 with 2 Axes>

In [12]:
import time
# Cargar una imagen o video
#source = "datasets/coco8/images/val/000000000036.jpg"
source = "datasets/coco128/images/val/000000000009.jpg"

# Medir tiempo de inferencia
t0 = time.time()
results = model(source)
t1 = time.time()

elapsed = t1 - t0
fps = 1 / elapsed
print(f"Inferencia en {elapsed:.3f}s → {fps:.2f} FPS")


image 1/1 C:\Users\menci\TFGMencia\PruebasModelos\datasets\coco128\images\val\000000000009.jpg: 480x640 4 bowls, 1 broccoli, 1 hot dog, 125.0ms
Speed: 2.0ms preprocess, 125.0ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)
Inferencia en 0.159s → 6.30 FPS


In [13]:
from ultralytics import YOLO
import time

# # -----------------------------
 # CONFIGURACIÓN
# # -----------------------------
MODEL_PATH = "runs/detect/train3/weights/best.pt"  # Ruta a tu modelo entrenado -> MIrar en el archivo de resultados correspondiente. 
#DATA_YAML = "coco8n.yaml"                           
DATA_YAML = "coco128.yaml"                          
EPOCHS = 50                                       
FPS_ITERATIONS = 10                               
#TEST_IMAGE = "datasets/coco8/images/val/000000000036.jpg" 
TEST_IMAGE = "datasets/coco128/images/val/000000000009.jpg"

# # INFO DEL MODELO
layers, params, grads, gflops = model.info()
params_m = params / 1e6

# # Extraer métricas del diccionario results_dict
res = metrics.results_dict
ap50 = res['metrics/mAP50(B)']
ap50_95 = res['metrics/mAP50-95(B)']
precision = res['metrics/precision(B)']
recall = res['metrics/recall(B)']


# Calcular F1-score 
f1_score = 2 * (precision * recall) / (precision + recall)

# # FPS
t_total = 0
for _ in range(FPS_ITERATIONS):
    t0 = time.time()
    _ = model(TEST_IMAGE)
    t_total += (time.time() - t0)
fps = FPS_ITERATIONS / t_total

# # IMPRIMIR RESULTADOS
print("\n========== METRICAS DEL MODELO ==========")
print(f"Epochs: {EPOCHS}")
print(f"Params(M): {params_m:.2f}")
print(f"GFLOPs: {gflops}")
print(f"FPS (bs=1): {fps:.2f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1_score:.3f}")
print(f"APval50: {ap50:.3f}")
print(f"APval50-95: {ap50_95:.3f}")
print("========================================\n")


Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

image 1/1 C:\Users\menci\TFGMencia\PruebasModelos\datasets\coco128\images\val\000000000009.jpg: 480x640 4 bowls, 1 broccoli, 1 hot dog, 68.5ms
Speed: 1.3ms preprocess, 68.5ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 C:\Users\menci\TFGMencia\PruebasModelos\datasets\coco128\images\val\000000000009.jpg: 480x640 4 bowls, 1 broccoli, 1 hot dog, 65.8ms
Speed: 1.4ms preprocess, 65.8ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 C:\Users\menci\TFGMencia\PruebasModelos\datasets\coco128\images\val\000000000009.jpg: 480x640 4 bowls, 1 broccoli, 1 hot dog, 64.9ms
Speed: 1.3ms preprocess, 64.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 C:\Users\menci\TFGMencia\PruebasModelos\datasets\coco128\images\val\000000000009.jpg: 480x640 4 bowls, 1 broccoli, 1 hot dog, 60.0ms
Speed: 1.0ms preprocess, 60.0ms inference, 1.0ms post

In [13]:
from ultralytics import YOLO
import numpy as np

# AP por clase y nombres
ap_per_class = metrics.box.ap            # numpy array de AP por clase
names_dict = metrics.names               # diccionario {indice: nombre}
class_indices = list(range(len(ap_per_class)))
class_names = [names_dict[i] for i in class_indices]

# Encontrar la clase con mayor AP usando numpy
max_index = np.argmax(ap_per_class)
max_ap = ap_per_class[max_index]
best_class = class_names[max_index]

print(f"La clase con mayor AP@0.5 es '{best_class}' con un AP de {max_ap:.3f}")


La clase con mayor AP@0.5 es 'horse' con un AP de 0.995
